# Colab A100 Heavy Search Pipeline: Top 40 Targets & Automated Teardown

This notebook implements the heavy search pipeline for the *Packing N Unit Squares into the Smallest Square* benchmark on an NVIDIA A100 GPU:
1. **GPU-Accelerated Screening**: Continuous angle and offset grid-screening using PyTorch on CUDA across the top 40 targets.
2. **Candidate Refinement**: Multi-variable 4-sign-piece epigraph LP relaxation using HiGHS.
3. **High-Precision Verification**: Direct certification of candidate packings against the official mpmath 50-digit interval metric.
4. **Automated Teardown**: Programmatic disconnect via `google.colab.runtime.unassign()` to prevent idle cloud billing costs.


In [ ]:
# 1. GPU Environment & CUDA Verification
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Device Total Memory: {mem_gb:.2f} GB")


In [ ]:
# 2. Dependency Installation
!pip install -q scipy mpmath numpy pandas


In [ ]:
# 3. Path Configuration
import os, sys
from pathlib import Path

REPO_DIR = Path.cwd()
# Add pipeline and baseline packages to sys.path
pipeline_dir = REPO_DIR / "poc/research/poc"
baseline_dir = REPO_DIR / "poc/baseline/poc/src"

for p in (pipeline_dir, baseline_dir):
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

print("Configured sys.path:")
for p in sys.path[:3]:
    print("  -", p)


In [ ]:
# 4. Pipeline Inspection & Top 40 Targets
from colab_runner import TOP_40_TARGETS, GPUScreener, run_pipeline, verify_candidates, teardown_colab_runtime

print(f"Loaded {len(TOP_40_TARGETS)} target instances for search.")
print("Targets:", TOP_40_TARGETS)

screener = GPUScreener()
print(f"Active Screener Device: {screener.device} (CUDA: {screener.is_cuda})")


In [ ]:
# 5. Parallel Global Search Across Top 40 Targets
output_json = REPO_DIR / "poc/research/data/colab_a100_candidates.json"

# Execute GPU batch screening and 4-sign-piece epigraph LP polish
pipeline_results = run_pipeline(
    targets=TOP_40_TARGETS,
    output_path=output_json,
    batch_size=256,
    teardown_after=False,
)


In [ ]:
# 6. Candidate Serialization to Competition CSV
import json, csv

with open(output_json, "r", encoding="utf-8") as f:
    data = json.load(f)

candidates = data.get("candidates", {})
csv_out = REPO_DIR / "poc/research/data/colab_a100_submission.csv"

with open(csv_out, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "n", "square_id", "x", "y", "angle"])
    row_id = 0
    for n_str in sorted(candidates.keys(), key=int):
        cand = candidates[n_str]
        layout = cand.get("layout", [])
        for sq in layout:
            writer.writerow([
                row_id,
                n_str,
                sq["id"],
                f"{sq['x']:.17g}",
                f"{sq['y']:.17g}",
                f"{sq['angle']:.17g}",
            ])
            row_id += 1

print(f"Exported {row_id} squares across {len(candidates)} targets to {csv_out}")


In [ ]:
# 7. Verification Against Official 50-Digit Certifier
cert_report_path = REPO_DIR / "poc/research/data/colab_a100_verification_report.json"
verification_results = verify_candidates(
    candidates_file=output_json,
    output_report=cert_report_path,
    dilation=1.0 + 5e-9,
)


### 8. Automated Cloud Runtime Teardown
To prevent ongoing cloud compute costs on Google Colab A100 instances, the cell below programmatically unassigns and disconnects the runtime upon completion.


In [ ]:
# 8. Automated Cloud Runtime Teardown / Self-Termination
# Set confirm_teardown = True to execute immediate runtime disconnection
confirm_teardown = True

if confirm_teardown:
    print("Disconnecting Google Colab A100 runtime to stop compute billing...")
    teardown_colab_runtime(dry_run=False)
else:
    print("Automated teardown bypassed.")
